In [6]:
import os
import sys

# 1. Uninstall potentially conflicting versions
!pip uninstall -y langchain langchain-classic langgraph langgraph-prebuilt

# 2. Install all dependencies
!pip install -q \
  "numpy<2.0" \
  websockets==11.0.3 \
  torch torchvision torchaudio \
  transformers>=4.38.0 \
  datasets \
  accelerate \
  evaluate \
  rouge_score \
  sentence-transformers \
  faiss-gpu-cu12 \
  langchain \
  langchain-community \
  langchain-huggingface \
  langchain-core \
  gradio==3.50.2

# 3. Automatic Restart Logic for NumPy fix
import numpy as np
if np.version.version.startswith('2.'):
    print(f"\n[INFO] NumPy version {np.__version__} detected. Downgrading and restarting session...")
    os._exit(0) # This triggers an automatic restart of the Colab runtime
else:
    print(f"\n[SUCCESS] Environment ready. NumPy version: {np.__version__}")

Found existing installation: langchain 1.2.13
Uninstalling langchain-1.2.13:
  Successfully uninstalled langchain-1.2.13
Found existing installation: langchain-classic 1.0.3
Uninstalling langchain-classic-1.0.3:
  Successfully uninstalled langchain-classic-1.0.3
Found existing installation: langgraph 1.1.3
Uninstalling langgraph-1.1.3:
  Successfully uninstalled langgraph-1.1.3
Found existing installation: langgraph-prebuilt 1.0.8
Uninstalling langgraph-prebuilt-1.0.8:
  Successfully uninstalled langgraph-prebuilt-1.0.8

[SUCCESS] Environment ready. NumPy version: 1.26.4


In [2]:
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# If getting value error in this cell, restart the session.

Device: cuda


In [3]:
print("************* Loading dataset *****************")
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train"
)

print("*********** Building knowledge base ***********")
kb_docs = [
    Document(
        page_content=r["response"],
        metadata={"intent": r["intent"]}
    )
    for r in dataset.select(range(3000))
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(kb_docs, embeddings)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


************* Loading dataset *****************


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


*********** Building knowledge base ***********


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
def preprocess(examples):

    inputs = [f"support_agent: {x}" for x in examples["instruction"]]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding=False
    )


    labels = tokenizer(
        text_target=examples["response"],
        max_length=256,
        truncation=True,
        padding=False
    )

    # Prepare label IDs (ignoring the pad tokens for loss calculation)
    labels_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

In [6]:
tokenized_data = (
    dataset
    .select(range(3000))
    .map(preprocess, batched=True)
    .train_test_split(test_size=0.1)
)


In [7]:
import torch
import gc
import numpy as np
import evaluate
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    Adafactor
)

# 1. Clear GPU memory and garbage collector to start fresh
torch.cuda.empty_cache()
gc.collect()

# 2. Setup Evaluation Metric
metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

# 3. Super Stability Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_support_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,

    # --- CRITICAL MEMORY REDUCTION ---
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    # ---------------------------------

    num_train_epochs=3,
    weight_decay=0.01,

    # --- NUMERICAL STABILITY  ---
    fp16=False,                     # Keep FP16 OFF to prevent NaN errors
    bf16=False,                     # T4 hardware cannot do BF16
    optim="adafactor",              # Native T5 optimizer for stability
    max_grad_norm=1.0,              # Prevent gradient explosions
    # ------------------------------------

    predict_with_generate=True,
    logging_steps=10,               # Frequent logging to monitor progress
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    report_to="none"
)

# 4. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics
)

# 5. Enable checkpointing on the model object explicitly
model.gradient_checkpointing_enable()

print("*********** Fine-tuning started (Super Stability Mode) **************")
trainer.train()

*********** Fine-tuning started (Super Stability Mode) **************


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,55.287689,6.169334,0.179900,0.063900,0.146700,0.146500
2,48.219235,5.277271,0.208900,0.087500,0.167000,0.167100
3,45.495804,5.043430,0.212800,0.092800,0.170100,0.170000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=507, training_loss=52.866347273425944, metrics={'train_runtime': 1418.5865, 'train_samples_per_second': 5.71, 'train_steps_per_second': 0.357, 'total_flos': 212887316109312.0, 'train_loss': 52.866347273425944, 'epoch': 3.0})

In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM

# 1. Save the best weights locally
trainer.save_model("./t5_support_final_best")
tokenizer.save_pretrained("./t5_support_final_best")

# 2. Clear memory again to be safe
torch.cuda.empty_cache()

# 3. Reload the model into 'model' variable
# This ensures run_agent uses the version that learned your policies
print("Loading fine-tuned weights...")
model = AutoModelForSeq2SeqLM.from_pretrained("./t5_support_final_best").to(device)
model.eval() # CRITICAL: Sets model to inference mode
print("✅ Model is now fine-tuned and ready!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading fine-tuned weights...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model is now fine-tuned and ready!


In [2]:
def run_agent(message, history=None):
    try:
        query = message.get("text", "") if isinstance(message, dict) else str(message)
        query = query.strip().lower()

        # Special business rule
        if "renew" in query.lower() and "subscription" in query.lower():
            return (
                "At the moment, I can help with newsletter subscriptions. "
                "For renewing paid subscriptions, please contact our associate."
            )

        # Escalation
        if any(k in query.lower() for k in ["sue", "legal", "lawyer", "court"]):
            return " Sorry for the inconvenience. Our Customer Support Executive will be contacting you shortly."

        # 1. Corrected Intent Map
        intent_map = {
            "cancel": ["cancel_order"],
            "track": ["order_status"],
            "where is my": ["order_status"],
            "replace": ["change_order"],
            "refund": ["get_refund"],
            "address": ["edit_account"],
            "password": ["registration_problems"],
        }

        detected_intent = None
        for k, v in intent_map.items():
            if k in query:
                detected_intent = v
                break

        # 2. Retrieval
        if detected_intent:
            filtered = [d for d in kb_docs if d.metadata.get("intent") in detected_intent]
            if filtered:
                temp_db = FAISS.from_documents(filtered, embeddings)
                docs = temp_db.similarity_search(query, k=3)
            else:
                docs = vector_db.similarity_search(query, k=5)
        else:
            docs = vector_db.similarity_search(query, k=5)

        # 3. Reranking with lower threshold
        pairs = [[query, d.page_content] for d in docs]
        scores = reranker.predict(pairs)
        best_idx = scores.argmax()

        if scores[best_idx] < -1.0: # Very permissive threshold
            return "I couldn't find a specific policy. How else can I help?"

        best_policy = docs[best_idx].page_content

        # 4. Improved Prompt
        prompt = (
            f"Task: Provide a professional answer using the Policy below. "
            f"If the policy requires an order number, ask the customer for it.\n\n"
            f"Policy: {best_policy}\n"
            f"Customer: {query}\n\n"
            f"Answer:"
        )

        # 5. Generation
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=120,
                num_beams=5,
                do_sample=True,
                temperature=0.8,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        return tokenizer.decode(output[0], skip_special_tokens=True).replace("Answer:", "").strip()

    except Exception as e:
        return f"Error: {e}"

In [4]:
import gradio as gr
gui = gr.ChatInterface(
    fn=run_agent,
    title="Intelligent Customer Support Chatbot V2",
    description="Hi, I'm AI Assistant. How can I help you?",
    examples=[
        "Cancel my order",
        "Replace my order",
        "Change delivery Address",
        "I want to escalate this issue"
    ]
)

gui.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://b0d2e512a4b3cb04ee.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b0d2e512a4b3cb04ee.gradio.live
